# 07 — Dashboard Exports

Gathers everything produced by notebooks 05-06 (plus a fresh copy of the feature-engineered dataset) and
exports it to `dashboard/data_exports/` for the Streamlit dashboard.

This is data movement, not modeling logic — nothing here needed to move into `src/`.

In [ ]:
import pandas as pd

from credit_risk.config import settings
from credit_risk.data import clean_and_label, load_raw_data
from credit_risk.features import build_features

## Rebuild the feature-engineered dataset

The full dataset isn't persisted between notebooks (see notebook 02) — rebuilding it here is a fast, pure
function chain, not a retrain.

In [ ]:
dashboard_df = build_features(clean_and_label(load_raw_data(settings.raw_data_path)))
dashboard_df.shape

## Load prediction artifacts from notebooks 05-06

In [ ]:
predictions_dir = settings.outputs_dir / "predictions"

risk_scores = pd.read_csv(predictions_dir / "risk_scores.csv")
threshold_df = pd.read_csv(predictions_dir / "threshold_metrics.csv")
feature_importance = pd.read_csv(predictions_dir / "feature_importance.csv")
portfolio_summary = pd.read_csv(predictions_dir / "portfolio_summary.csv")
decision_default_rate = pd.read_csv(predictions_dir / "decision_default_rate.csv")

In [ ]:
risk_band_analysis = (risk_scores.groupby("risk_band")["actual"].mean() * 100).reset_index()
risk_band_analysis.columns = ["risk_band", "default_rate"]
risk_band_analysis

## Export

In [ ]:
export_dir = settings.dashboard_export_dir
export_dir.mkdir(parents=True, exist_ok=True)

dashboard_df.to_csv(export_dir / "dashboard_dataset.csv", index=False)
risk_scores.to_csv(export_dir / "risk_scores.csv", index=False)
threshold_df.to_csv(export_dir / "threshold_metrics.csv", index=False)
feature_importance.to_csv(export_dir / "feature_importance.csv", index=False)
portfolio_summary.to_csv(export_dir / "portfolio_summary.csv", index=False)
decision_default_rate.to_csv(export_dir / "decision_default_rate.csv", index=False)
risk_band_analysis.to_csv(export_dir / "risk_band_analysis.csv", index=False)

In [ ]:
import os

sorted(os.listdir(export_dir))

Streamlit dashboard itself doesn't exist yet — these exports are ready for it once that dashboard gets
built.